In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


# Load the data again for visualization purposes
data = pd.read_csv('person.csv') 

# Remove all rows with ages 999 and 998 from the dataset
cleaned_data = data[(data['AGE'] != 999) & (data['AGE'] != 998)]

male_female_counts = cleaned_data[(cleaned_data['SEX'] != 8) & (cleaned_data['SEX'] != 9)]

columns_with_nan_except_age_sex = [
    'RUR_URB', 'FUNC_SYS', 'MAKE', 'MAK_MOD', 'BODY_TYP', 'MOD_YEAR', 'TOW_VEH',
    'SPEC_USE', 'EMER_USE', 'ROLLOVER', 'IMPACT1', 'FIRE_EXP', 'ROAD_FNC',
    'CERT_NO', 'VINTYPE', 'VINMAKE', 'VINA_MOD', 'VIN_BT', 'VINMODYR', 'VIN_LNGT',
    'VIN_WGT', 'WGTCD_TR', 'WHLBS_LG', 'WHLBS_SH', 'SER_TR', 'FUELCODE',
    'MCYCL_DS', 'CARBUR', 'CYLINDER', 'DISPLACE', 'MCYCL_CY', 'TIRE_SZE', 'TON_RAT',
    'TRK_WT', 'TRKWTVAR', 'MCYCL_WT', 'VIN_REST', 'WHLDRWHL'
]


# Removing all the above columns with NaN values (except for 'AGE' and 'SEX') from the cleaned dataset
cleaned_data_dropped_nans = male_female_counts.drop(columns=columns_with_nan_except_age_sex)


In [ ]:
cleaned_data_dropped_nans['AgeBand'] = pd.cut(cleaned_data_dropped_nans['AGE'], 10, labels=[0,1,2,3,4,5,6,7,8,9])
cleaned_data_dropped_nans[['AgeBand', 'INJ_SEV']].groupby(['AgeBand'], as_index=False).median().sort_values(by='AgeBand', ascending=True)

In [ ]:
cleaned_data_dropped_nans[["ALC_DET", "INJ_SEV"]].groupby(['ALC_DET'], as_index=False).mean().sort_values(by='INJ_SEV', ascending=False)

In [ ]:
cleaned_data_dropped_nans[["DRINKING", "INJ_SEV"]].groupby(['DRINKING'], as_index=False).mean().sort_values(by='INJ_SEV', ascending=False)

In [ ]:
accident_data = pd.read_csv('accident.csv')
# ROAD_FNC
accident_data = accident_data[['accident_id' ,'ST_CASE','HARM_EV', 'LGT_COND', 'WEATHER', 'A_DOW', 'A_ROADFC', 'COUNTY', 'CITY', 'PERSONS', 'MAN_COLL', 'A_POSBAC']]
merged_df = pd.merge(cleaned_data_dropped_nans, accident_data, on='accident_id', how='left', suffixes=('_person', '_accident'))
# print(merged_df)
merged_df.to_csv("merged.csv")
merged_df.dropna(subset=['ST_CASE_accident'], inplace=True)

# kaam_ke_columns = ['HARM_EV', 'LGT_COND', 'WEATHER', 'A_DOW', 'A_ROADFC', 'COUNTY', 'CITY', 'PERSONS', 'MAN_COLL', 'ROAD_FNC', 'A_POSBAC']
print(merged_df)

In [ ]:
merged_df[["WEATHER", "INJ_SEV"]].groupby(['WEATHER'], as_index=False).mean().sort_values(by='INJ_SEV', ascending=False)

In [ ]:
merged_df[["HARM_EV_accident", "INJ_SEV"]].groupby(['HARM_EV_accident'], as_index=False).mean().sort_values(by='INJ_SEV', ascending=False)

In [ ]:
from matplotlib import pyplot as plt
merged_df['HARM_EV_accident'].plot(kind='hist', bins=20, title='HARM_EV_accident')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
from sklearn.model_selection import train_test_split
# cleaned_data_final = merged_df
# 'DRUGS', 'DSTATUS', 'DRINKING', 'AIR_BAG', 'ALC_STATUS'
columns_to_remove = ['index', 'accident_id', 'STATE', 'VEH_NO', 'STR_VEH', 'DAY', 'MINUTE', 'SCH_BUS',
                     'AGE', 'SEX', 'PER_TYP', 'SEAT_POS', 'REST_USE', 'REST_MIS', 'EJECTION', 'EJ_PATH', 'EXTRICAT',
                    'ATST_TYP', 'DRUGTST1', 'DRUGTST2', 'ST_CASE_person', 'ST_CASE_accident', 'HARM_EV_accident',
                     'COUNTY_accident', 'CITY',
                     'DRUGTST3', 'DRUGRES1', 'DRUGRES2', 'DRUGRES3', 'DOA', 'DEATH_DA', 'DEATH_MO', 'DEATH_YR',
                     'DEATH_HR', 'DEATH_MN', 'DEATH_TM', 'LAG_HRS', 'LAG_MINS', 'P_SF1', 'P_SF2', 'P_SF3', 'HISPANIC', 'WORK_INJ',
                     'RACE', 'LOCATION', 'Year', 'MAN_COLL_accident']
cleaned_data_final = merged_df.drop(columns_to_remove, axis=1)

In [ ]:
corrs = {}
for column in cleaned_data_final:
  corrs[column] = cleaned_data_final['INJ_SEV'].corr(cleaned_data_final[column], method='spearman')

sorted_dict = sorted(corrs.items(), key=lambda item: item[1])

corr_df = pd.DataFrame(sorted_dict)
print(corr_df)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Assuming `cleaned_data_final` is your DataFrame and 'INJ_SEV' is the target variable
X = cleaned_data_final.drop('INJ_SEV', axis=1)
y = cleaned_data_final['INJ_SEV']

# Initialize the model
model = RandomForestClassifier()

# Fit the model
model.fit(X, y)

# Get feature importances
importances = model.feature_importances_

# Sort the feature importances in descending order and print them
feature_importances = sorted(zip(X.columns, importances), key=lambda x: x[1], reverse=True)
for feature, importance in feature_importances:
    print(f'{feature}: {importance}')

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Initialize the base classifier
model = LogisticRegression(C=0.5, max_iter=10000)

# Initialize RFE and select the top 10 features, for example
rfe = RFE(model, n_features_to_select=10)

# scaler = StandardScaler()

# Scale the features
# X_scaled = scaler.fit_transform(X)

# Fit RFE
rfe.fit(X, y)

# Print the ranking of features
for i in range(X.shape[1]):
    print(f'{X.columns[i]}: {rfe.ranking_[i]}')

In [ ]:
# Assuming 'INJ_SEV' is the target variable and the rest are features
X = cleaned_data_final.drop('INJ_SEV', axis=1)
y = cleaned_data_final['INJ_SEV']
cleaned_data_final.to_csv("merged.csv")
# X_train = X
# Y_train = y

# Splitting the dataset into training and testing sets
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=4)
X_train

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

In [ ]:
### Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Create a pipeline with preprocessing and model
pipeline = make_pipeline(StandardScaler(), LogisticRegression(solver='liblinear', random_state=42))

# Define the parameter grid
param_grid_lr = {
    'logisticregression__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'logisticregression__penalty': ['l1', 'l2']  # Including different penalties
}

# Initialize GridSearchCV with StratifiedKFold
grid_search_lr = GridSearchCV(pipeline, param_grid_lr, cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1)

# Fit GridSearchCV to the training data
grid_search_lr.fit(X_train, Y_train)
logreg_train = round(grid_search_lr.best_score_ * 100, 2)

# Evaluate the model on the test set
logreg_test = round(grid_search_lr.score(X_test, Y_test) * 100, 2)
Y_pred = grid_search_lr.predict(X_test)

# Print the best parameters, the best training score and the test score
print("Best Parameters:", grid_search_lr.best_params_)
print("Best Score:", grid_search_lr.best_score_)
print("Training Accuracy:", logreg_train)
print("Test Accuracy:", logreg_test)

# Calculate F1-score and precision
f1_score_lr = f1_score(Y_test, Y_pred, average='macro')
precision_score_lr = precision_score(Y_test, Y_pred, average='macro')
recall_score_lr = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_lr)
print("Precision Score:", precision_score_lr)
print("Recall Score:", recall_score_lr)

In [ ]:
### Support Vector Machines
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score

# Initialize a pipeline with scaling and SVM
pipeline = make_pipeline(StandardScaler(), SVC())

# Define the parameter grid
param_grid = {
    'svc__C': [0.1, 1, 10, 100],  # Regularization parameter
    'svc__gamma': ['scale', 'auto', 0.01, 0.1, 1, 10],  # Kernel coefficient
    'svc__kernel': ['linear', 'poly', 'rbf', 'sigmoid']  # Kernel type
}

# Initialize GridSearchCV with a more robust CV strategy
grid_search_svm = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)

# Assuming X_train, Y_train, X_test, Y_test are your datasets
# Fit GridSearchCV to the training data
grid_search_svm.fit(X_train, Y_train)

# Print the best parameters and the best score
print("Best Parameters:", grid_search_svm.best_params_)
print("Best Score:", grid_search_svm.best_score_)

# Use the best estimator to make predictions
Y_pred_svm = grid_search_svm.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy_svm = accuracy_score(Y_test, Y_pred_svm)
print("Test Accuracy:", test_accuracy_svm)

svm_train = round(grid_search_svm.best_score_ * 100, 2)
svm_test = round(test_accuracy_svm * 100, 2)

# Calculate F1-score and precision
f1_score_svm = f1_score(Y_test, Y_pred_svm, average='macro')
precision_score_svm = precision_score(Y_test, Y_pred_svm, average='macro')
recall_score_svm = recall_score(Y_test, Y_pred_svm, average='macro')
print("F1 Score:", f1_score_svm)
print("Precision Score:", precision_score_svm)
print("Recall Score:", recall_score_svm)

In [ ]:
### Random Forest
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300, 400, 500],  # Expanded range of trees
    'max_depth': [None, 10, 20, 30, 40, 50],  # Increased depth options
    'min_samples_split': [2, 5, 10, 20],  # Increased granularity
    'min_samples_leaf': [1, 2, 4, 6],  # More options for leaf nodes
    'bootstrap': [True, False],  # Whether to bootstrap samples
    'class_weight': [None, 'balanced', 'balanced_subsample']  # Handling class imbalance
}

# Initialize the model
rf = RandomForestClassifier(random_state=42)

# Initialize GridSearchCV or RandomizedSearchCV
grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit GridSearchCV to the training data
grid_search_rf.fit(X_train, Y_train)

# Print the best parameters and the best score
print("Best Parameters:", grid_search_rf.best_params_)
print("Best Score:", grid_search_rf.best_score_)

# Use the best estimator to make predictions
best_rf = grid_search_rf.best_estimator_
Y_pred = best_rf.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy_rf = accuracy_score(Y_test, Y_pred)
print("Test Accuracy:", test_accuracy_rf)

rf_train = round(grid_search_rf.best_score_ * 100, 2)
rf_test = round(test_accuracy_rf * 100, 2)
print(f"Training Accuracy: {rf_train}%")
print(f"Test Accuracy: {rf_test}%")

# Calculate F1-score and precision
f1_score_rf = f1_score(Y_test, Y_pred, average='macro')
precision_score_rf = precision_score(Y_test, Y_pred, average='macro')
recall_score_rf = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_rf)
print("Precision Score:", precision_score_rf)
print("Recall Score:", recall_score_rf)


In [ ]:
### Decision Tree
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

# Assuming X_train, Y_train, X_test, Y_test are your datasets

# Initialize the StandardScaler
scaler = StandardScaler()

# Scale the features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the model
decision_tree = DecisionTreeClassifier(random_state=42)

# Define the parameter grid
param_grid = {
    'max_depth': [None, 10, 15, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 15, 20],
    'min_samples_leaf': [1, 2, 5, 10, 15],
    'ccp_alpha': [0.0, 0.01, 0.05, 0.1],
    'class_weight': [None, 'balanced']
}

# Setup Grid Search
grid_search_dt = GridSearchCV(decision_tree, param_grid, cv=10, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the data
grid_search_dt.fit(X_train_scaled, Y_train)
acc_grid_search_dt = round(grid_search_dt.score(X_train_scaled, Y_train) * 100, 2)

# Print training accuracy and best parameters
print(f"Training Accuracy: {acc_grid_search_dt}%")
print("Best Parameters:", grid_search_dt.best_params_)

# Predict and calculate the accuracy on the test set
Y_pred = grid_search_dt.predict(X_test_scaled)
test_accuracy_dt = accuracy_score(Y_test, Y_pred)
dt_test = round(test_accuracy_dt * 100, 2)
dt_train = round(grid_search_dt.best_score_ * 100, 2)

print("Test Accuracy:", dt_test)

# Calculate F1-score and precision
f1_score_dt = f1_score(Y_test, Y_pred, average='macro')
precision_score_dt = precision_score(Y_test, Y_pred, average='macro')
recall_score_dt = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_dt)
print("Precision Score:", precision_score_dt)
print("Recall Score:", recall_score_dt)


In [ ]:
### SGD
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Define a pipeline to scale data and then apply SGD
pipeline = make_pipeline(StandardScaler(), SGDClassifier(max_iter=1000, tol=1e-3, random_state=42))

# Define the parameter grid
param_grid = {
    'sgdclassifier__alpha': [0.0001, 0.0005, 0.001, 0.005, 0.01],
    'sgdclassifier__penalty': ['l2', 'l1', 'elasticnet'],
    'sgdclassifier__learning_rate': ['constant', 'optimal', 'invscaling', 'adaptive'],
    'sgdclassifier__eta0': [0.01, 0.05, 0.1],
    'sgdclassifier__loss': ['hinge', 'log', 'modified_huber'],  # Varying loss functions
    'sgdclassifier__early_stopping': [True],  # Enable early stopping
    'sgdclassifier__n_iter_no_change': [5, 10]  # Iterations with no improvement
}

# Initialize GridSearchCV with stratified k-folds
grid_search_sgd = GridSearchCV(
    pipeline, param_grid, cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1
)

# Fit GridSearchCV to the training data
grid_search_sgd.fit(X_train, Y_train)

# Print the best parameters and the best score
print("Best Parameters:", grid_search_sgd.best_params_)
print("Best Score:", grid_search_sgd.best_score_)

Y_pred = grid_search_dt.predict(X_test)

sgd_train = round(grid_search_sgd.best_score_ * 100, 2)
sgd_test = round(grid_search_sgd.score(X_test, Y_test) * 100, 2)
print("Training Accuracy:", sgd_train)
print("Test Accuracy:", sgd_test)

# Calculate F1-score and precision
f1_score_sgd = f1_score(Y_test, Y_pred, average='macro')
precision_score_sgd = precision_score(Y_test, Y_pred, average='macro')
recall_score_sgd = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_sgd)
print("Precision Score:", precision_score_sgd)
print("Recall Score:", recall_score_sgd)

In [ ]:
#### Linear SVC
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Define a pipeline to scale data and then apply LinearSVC
pipeline = make_pipeline(StandardScaler(), LinearSVC(random_state=42))

# Define the parameter grid with conditional settings
param_grid_lsvc = [
    {'linearsvc__C': [0.001, 0.01, 0.1, 1, 10, 100],
     'linearsvc__loss': ['squared_hinge'],  # Safe choice for dual=False
     'linearsvc__dual': [False],
     'linearsvc__class_weight': [None, 'balanced']},
    {'linearsvc__C': [0.001, 0.01, 0.1, 1, 10, 100],
     'linearsvc__loss': ['hinge'],  # Typical with dual=True
     'linearsvc__dual': [True],
     'linearsvc__class_weight': [None, 'balanced']}
]

# Initialize GridSearchCV with stratified k-folds
grid_search_lsvc = GridSearchCV(
    pipeline, param_grid_lsvc, cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1
)

# Fit GridSearchCV to the training data
grid_search_lsvc.fit(X_train, Y_train)

linear_svc_train = round(grid_search_lsvc.best_score_ * 100, 2)
linear_svc_test = round(grid_search_lsvc.score(X_test, Y_test) * 100, 2)
Y_pred = grid_search_lsvc.predict(X_test)

# Print the best parameters and the best score
print("Best Parameters:", grid_search_lsvc.best_params_)
print("Best Score:", grid_search_lsvc.best_score_)
print("Test Accuracy:", linear_svc_test)

# Calculate F1-score and precision
f1_score_lsvc = f1_score(Y_test, Y_pred, average='macro')
precision_score_lsvc = precision_score(Y_test, Y_pred, average='macro')
recall_score_lsvc = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_lsvc)
print("Precision Score:", precision_score_lsvc)
print("Recall Score:", recall_score_lsvc)

In [ ]:
### Multilayer Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline for scaling and model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(random_state=42))
])

# Parameter grid for GridSearchCV
param_grid = {
    'mlp__hidden_layer_sizes': [(50,), (100,), (50, 50)],  # Example: one layer of 50, one of 100, two layers of 50
    'mlp__activation': ['tanh', 'relu'],
    'mlp__alpha': [0.0001, 0.001, 0.01],  # Regularization term
    'mlp__max_iter': [100, 150]  # Number of epochs
}

# GridSearchCV for hyperparameter tuning
grid_search_mlp = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fitting GridSearchCV
grid_search_mlp.fit(X_train, Y_train)
mlp_test = round(grid_search_mlp.score(X_test, Y_test) * 100, 2)
Y_pred = grid_search_mlp.predict(X_test)

# Best parameters and score
print('Best Parameters:', grid_search_mlp.best_params_)
print('Best Score:', grid_search_mlp.best_score_)
print('Test Accuracy:', mlp_test)
mlp_train = round(grid_search_mlp.best_score_ * 100, 2)

# Calculate F1-score and precision
f1_score_mlp = f1_score(Y_test, Y_pred, average='macro')
precision_score_mlp = precision_score(Y_test, Y_pred, average='macro')
recall_score_mlp = recall_score(Y_test, Y_pred, average='macro')
print("F1 Score:", f1_score_mlp)
print("Precision Score:", precision_score_mlp)
print("Recall Score:", recall_score_mlp)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Creating a label encoder object
le = LabelEncoder()

# Fitting the encoder to the full class range (assuming you know all potential classes)
le.fit([0, 1, 2, 3, 4, 5, 9])

# Transforming the labels in both training and test datasets
Y_train_encoded = le.transform(Y_train)
Y_test_encoded = le.transform(Y_test)

In [ ]:
### XGBoost
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import label_binarize
import numpy as np

# Example dataset placeholders
# X_train, Y_train, X_test, Y_test = your_training_and_testing_sets

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 6, 9],
    'min_child_weight': [1, 2, 4],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9]
}

xgb = XGBClassifier(eval_metric='logloss', enable_categorical=True)

# Initialize GridSearchCV
grid_search_xgb = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=3, n_jobs=-1, verbose=2, scoring='accuracy')

# Fit GridSearchCV to the training data
grid_search_xgb.fit(X_train, Y_train_encoded)

# Print the best parameters and the best score
print("Best Parameters:", grid_search_xgb.best_params_)
print("Best Score:", grid_search_xgb.best_score_)

# Use the best estimator to make predictions
best_xgb = grid_search_xgb.best_estimator_
Y_pred = best_xgb.predict(X_test)

# Calculate accuracy on the test set
test_accuracy_xgb = accuracy_score(Y_test_encoded, Y_pred)
print("Test Accuracy:", test_accuracy_xgb)
xgb_train = round(grid_search_xgb.best_score_ * 100, 2)
xgb_test = round(test_accuracy_xgb * 100, 2)

# Calculate F1-score and precision
f1_score_xgb = f1_score(Y_test_encoded, Y_pred, average='macro')
precision_score_xgb = precision_score(Y_test_encoded, Y_pred, average='macro')
recall_score_xgb = recall_score(Y_test_encoded, Y_pred, average='macro')
print("F1 Score:", f1_score_xgb)
print("Precision Score:", precision_score_xgb)
print("Recall Score:", recall_score_xgb)

In [ ]:
models = pd.DataFrame({
    'Model': ['Logistic Regression', 'Support Vector Machines', 'Random Forest',
              'Decision Tree', 'Stochastic Gradient Decent', 'Linear SVC', 'Multilayer Perceptron', 'XGBoost'],
    'Train Score': [logreg_train, svm_train, rf_train, dt_train, sgd_train, linear_svc_train, mlp_train, xgb_train],
    'Test Score': [logreg_test, svm_test, rf_test, dt_test, sgd_test, linear_svc_test, mlp_test, xgb_test],
    'Precision Score': [precision_score_lr, precision_score_svm, precision_score_rf, precision_score_dt, precision_score_sgd, precision_score_lsvc, precision_score_mlp, precision_score_xgb],
    'Recall Score': [recall_score_lr,recall_score_svm, recall_score_rf,recall_score_dt,recall_score_sgd, recall_score_lsvc, recall_score_mlp, recall_score_xgb],
    'F1 Score': [f1_score_lr, f1_score_svm, f1_score_rf, f1_score_dt, f1_score_sgd, f1_score_lsvc, f1_score_mlp, f1_score_xgb]
    })
models.sort_values(by='Test Score', ascending=False)